# Health Insurance Fraud Detection — Feature Analysis

## Objective

This notebook converts the exploratory analysis into a **production-oriented feature specification**.

Goals:

- prevent target and temporal leakage;
- define model-eligible features;
- derive defensible temporal and business features;
- handle missingness explicitly;
- audit redundancy;
- freeze temporal train / validation / test splits;
- prepare a reproducible scikit-learn preprocessing pipeline;
- define the final modeling contract.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "interim"

claims = pd.read_parquet(DATA_DIR / "claims.parquet")

print(f"Claims: {len(claims):,}")
print(f"Columns: {claims.shape[1]}")
print(f"Fraud prevalence: {claims['is_fraud'].mean():.3%}")

Claims: 99,911
Columns: 55
Fraud prevalence: 2.504%


## 1. Feature governance

In [2]:
TARGET = "is_fraud"

IDENTIFIER_COLUMNS = [
    "claim_id",
    "customer_id",
    "policy_id",
    "provider_id",
]

LEAKAGE_COLUMNS = [
    "legitimate_anomaly",
    "legitimate_anomaly_type",
    "latent_fraud_score",
    "synthetic_fraud_probability",
    "fraud_difficulty",
    "fraud_mechanism",
]

SYNTHETIC_PROFILE_COLUMNS = [
    "customer_behavior_segment",
    "provider_behavior_segment",
]

RAW_DATE_COLUMNS = [
    "service_date",
    "claim_submission_date",
    "claim_submission_timestamp",
]

EXCLUDED_COLUMNS = (
    IDENTIFIER_COLUMNS
    + LEAKAGE_COLUMNS
    + SYNTHETIC_PROFILE_COLUMNS
    + [TARGET]
)

base_candidate_features = [
    c for c in claims.columns
    if c not in EXCLUDED_COLUMNS
]

print(f"Candidate columns before refinement: {len(base_candidate_features)}")

Candidate columns before refinement: 42


### Governance rationale

- **Identifiers** remain useful for joins and tracing but are excluded from the first predictive model.
- **Synthetic ground-truth variables** are forbidden because they reveal the data-generation process.
- **Synthetic profile variables** are excluded from the main model because they are latent simulation constructs.
- **Raw dates** are transformed into operational calendar features rather than passed directly.


## 2. Temporal availability audit

In [3]:
TEMPORAL_HISTORY_FEATURES = [
    "customer_claims_7d",
    "customer_claims_30d",
    "customer_claims_90d",
    "customer_claims_365d",
    "customer_amount_30d",
    "customer_amount_365d",
    "customer_avg_claim_amount_365d",
    "days_since_customer_previous_claim",
    "days_since_same_provider_claim",
    "customer_provider_claims_30d",
    "same_service_claims_30d",
    "provider_claims_30d",
    "provider_claims_90d",
    "provider_avg_claim_amount_90d",
    "service_typical_amount",
    "claim_to_service_median_ratio",
    "claim_to_customer_avg_ratio",
    "claim_to_provider_avg_ratio",
]

temporal_audit = pd.DataFrame({
    "feature": TEMPORAL_HISTORY_FEATURES,
    "available_at_scoring": True,
    "strict_past_required": True,
    "status": "eligible_if_prior_events_only",
})

temporal_audit

,feature,available_at_scoring,strict_past_required,status
0,customer_claims_7d,True,True,eligible_if_prior_events_only
1,customer_claims_30d,True,True,eligible_if_prior_events_only
2,customer_claims_90d,True,True,eligible_if_prior_events_only
3,customer_claims_365d,True,True,eligible_if_prior_events_only
4,customer_amount_30d,True,True,eligible_if_prior_events_only
5,customer_amount_365d,True,True,eligible_if_prior_events_only
6,customer_avg_claim_amount_365d,True,True,eligible_if_prior_events_only
7,days_since_customer_previous_claim,True,True,eligible_if_prior_events_only
8,days_since_same_provider_claim,True,True,eligible_if_prior_events_only
9,customer_provider_claims_30d,True,True,eligible_if_prior_events_only


Historical variables are eligible only because they were constructed from events **strictly earlier** than the current claim timestamp.

This anti-leakage condition must remain true in production.


## 3. Missingness strategy

In [4]:
modeling_missing = (
    claims[base_candidate_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

modeling_missing[modeling_missing > 0].to_frame("missing_pct")

,missing_pct
days_since_same_provider_claim,99.0772
days_since_policy_change,91.8978
claim_to_customer_avg_ratio,30.1638
customer_avg_claim_amount_365d,30.1638
days_since_customer_previous_claim,19.4453
has_prescription,2.9897
provider_avg_claim_amount_90d,2.1879
claim_to_provider_avg_ratio,2.1879
document_count,0.4644


In [5]:
claims["has_prescription_missing"] = claims["has_prescription"].isna().astype("int8")
claims["document_count_missing"] = claims["document_count"].isna().astype("int8")
claims["provider_missing"] = claims["provider_id"].isna().astype("int8")

[
    "has_prescription_missing",
    "document_count_missing",
    "provider_missing",
]

['has_prescription_missing', 'document_count_missing', 'provider_missing']

Missingness will not be globally filled before splitting.

All learned imputations must be fitted on the **training set only**.


## 4. Calendar feature engineering

In [6]:
claims["submission_hour"] = claims["claim_submission_timestamp"].dt.hour
claims["submission_dayofweek"] = claims["claim_submission_timestamp"].dt.dayofweek
claims["submission_month"] = claims["claim_submission_timestamp"].dt.month
claims["submission_is_weekend"] = (claims["submission_dayofweek"] >= 5).astype("int8")

claims["service_dayofweek"] = claims["service_date"].dt.dayofweek
claims["service_month"] = claims["service_date"].dt.month
claims["service_is_weekend"] = (claims["service_dayofweek"] >= 5).astype("int8")

DATE_DERIVED_FEATURES = [
    "submission_hour",
    "submission_dayofweek",
    "submission_month",
    "submission_is_weekend",
    "service_dayofweek",
    "service_month",
    "service_is_weekend",
]

claims[DATE_DERIVED_FEATURES].head()

,submission_hour,submission_dayofweek,submission_month,submission_is_weekend,service_dayofweek,service_month,service_is_weekend
0,0,6,1,1,4,12,0
1,1,6,1,1,3,12,0
2,1,6,1,1,0,12,0
3,2,6,1,1,4,12,0
4,2,6,1,1,4,12,0


## 5. Business-derived contextual features

In [7]:
EPS = 1e-6

claims["requested_to_limit_ratio"] = (
    claims["requested_reimbursement"]
    / claims["coverage_limit"].clip(lower=EPS)
)

claims["amount_above_service_typical"] = (
    claims["claim_amount"]
    - claims["service_typical_amount"]
)

claims["recent_claim_share_30d_365d"] = (
    claims["customer_claims_30d"]
    / claims["customer_claims_365d"].replace(0, np.nan)
)

claims["recent_amount_share_30d_365d"] = (
    claims["customer_amount_30d"]
    / claims["customer_amount_365d"].replace(0, np.nan)
)

claims["provider_recent_activity_ratio"] = (
    (claims["provider_claims_30d"] + 1)
    / (claims["provider_claims_90d"] / 3 + 1)
)

claims["customer_provider_intensity"] = (
    claims["customer_provider_claims_30d"]
    / (claims["customer_claims_30d"] + 1)
)

claims["same_service_intensity"] = (
    claims["same_service_claims_30d"]
    / (claims["customer_claims_30d"] + 1)
)

claims["days_since_policy_change_missing"] = (
    claims["days_since_policy_change"].isna().astype("int8")
)

DERIVED_BUSINESS_FEATURES = [
    "requested_to_limit_ratio",
    "amount_above_service_typical",
    "recent_claim_share_30d_365d",
    "recent_amount_share_30d_365d",
    "provider_recent_activity_ratio",
    "customer_provider_intensity",
    "same_service_intensity",
    "days_since_policy_change_missing",
]

claims[DERIVED_BUSINESS_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
requested_to_limit_ratio,"99,911.0000",0.3991,0.2455,0.0111,0.2102,0.3384,0.5316,1.0000
amount_above_service_typical,"99,911.0000",26.2134,153.3101,"-1,282.9400",-34.1500,-0.3400,50.5625,"3,218.8400"
recent_claim_share_30d_365d,"69,774.0000",0.1234,0.2741,0.0000,0.0000,0.0000,0.0000,1.0000
recent_amount_share_30d_365d,"69,774.0000",0.1227,0.2833,0.0000,0.0000,0.0000,0.0000,1.0000
provider_recent_activity_ratio,"99,911.0000",1.0349,0.3990,0.0811,0.7500,1.0000,1.2857,2.6250
customer_provider_intensity,"99,911.0000",0.0035,0.0462,0.0000,0.0000,0.0000,0.0000,0.8750
same_service_intensity,"99,911.0000",0.0202,0.1034,0.0000,0.0000,0.0000,0.0000,0.8750
days_since_policy_change_missing,"99,911.0000",0.9190,0.2729,0.0000,1.0000,1.0000,1.0000,1.0000


### Business rationale

These variables represent **relative context** rather than raw scale:

- reimbursement relative to the policy limit;
- recent activity relative to long-term activity;
- provider acceleration;
- concentration of claims within the same provider or service;
- amount deviation from the service baseline.

They are designed to be understandable to investigators.


## 6. Final candidate schema before redundancy review

In [8]:
RAW_MODEL_EXCLUSIONS = (
    IDENTIFIER_COLUMNS
    + LEAKAGE_COLUMNS
    + SYNTHETIC_PROFILE_COLUMNS
    + RAW_DATE_COLUMNS
    + [TARGET]
)

candidate_features = [
    c for c in claims.columns
    if c not in RAW_MODEL_EXCLUSIONS
]

print(f"Candidate features after transformations: {len(candidate_features)}")

Candidate features after transformations: 57


## 7. Feature type families

In [9]:
BOOLEAN_FEATURES = [
    c for c in candidate_features
    if (
        pd.api.types.is_bool_dtype(claims[c])
        or c.endswith("_missing")
        or c.endswith("_is_weekend")
        or c == "provider_missing"
    )
]

CATEGORICAL_FEATURES = [
    c for c in candidate_features
    if (
        pd.api.types.is_object_dtype(claims[c])
        or isinstance(claims[c].dtype, pd.CategoricalDtype)
    )
]

NUMERIC_FEATURES = [
    c for c in candidate_features
    if pd.api.types.is_numeric_dtype(claims[c])
    and c not in BOOLEAN_FEATURES
]

summary = pd.DataFrame({
    "family": ["numeric", "categorical", "boolean"],
    "count": [
        len(NUMERIC_FEATURES),
        len(CATEGORICAL_FEATURES),
        len(BOOLEAN_FEATURES),
    ],
})

summary

,family,count
0,numeric,42
1,categorical,7
2,boolean,8


In [10]:
print("NUMERIC FEATURES")
print(NUMERIC_FEATURES)

print("\nCATEGORICAL FEATURES")
print(CATEGORICAL_FEATURES)

print("\nBOOLEAN / INDICATOR FEATURES")
print(BOOLEAN_FEATURES)

NUMERIC FEATURES
['service_units', 'claim_amount', 'requested_reimbursement', 'coverage_limit', 'document_count', 'customer_age', 'customer_tenure_months', 'policy_tenure_months', 'days_since_policy_change', 'provider_tenure_months', 'days_service_to_submission', 'reimbursement_ratio', 'customer_claims_7d', 'customer_claims_30d', 'customer_claims_90d', 'customer_claims_365d', 'customer_amount_30d', 'customer_amount_365d', 'customer_avg_claim_amount_365d', 'days_since_customer_previous_claim', 'days_since_same_provider_claim', 'customer_provider_claims_30d', 'same_service_claims_30d', 'provider_claims_30d', 'provider_claims_90d', 'provider_avg_claim_amount_90d', 'service_typical_amount', 'claim_to_service_median_ratio', 'claim_to_customer_avg_ratio', 'claim_to_provider_avg_ratio', 'submission_hour', 'submission_dayofweek', 'submission_month', 'service_dayofweek', 'service_month', 'requested_to_limit_ratio', 'amount_above_service_typical', 'recent_claim_share_30d_365d', 'recent_amount_sh

## 8. Redundancy audit

In [11]:
numeric_corr = claims[NUMERIC_FEATURES].corr(method="spearman")

upper = np.triu(np.ones(numeric_corr.shape), k=1).astype(bool)

corr_pairs = (
    numeric_corr
    .where(upper)
    .stack()
    .rename("spearman_corr")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)

corr_pairs["abs_corr"] = corr_pairs["spearman_corr"].abs()

high_corr_pairs = (
    corr_pairs.loc[corr_pairs["abs_corr"] >= 0.90]
    .sort_values("abs_corr", ascending=False)
)

high_corr_pairs

,feature_1,feature_2,spearman_corr,abs_corr
669,customer_provider_claims_30d,customer_provider_intensity,1.0000,1.0000
689,same_service_claims_30d,same_service_intensity,0.9999,0.9999
851,recent_claim_share_30d_365d,recent_amount_share_30d_365d,0.9974,0.9974
457,customer_claims_30d,customer_amount_30d,0.9964,0.9964
478,customer_claims_30d,recent_claim_share_30d_365d,0.9893,0.9893
479,customer_claims_30d,recent_amount_share_30d_365d,0.9883,0.9883
557,customer_amount_30d,recent_amount_share_30d_365d,0.9877,0.9877
556,customer_amount_30d,recent_claim_share_30d_365d,0.9835,0.9835
41,claim_amount,requested_reimbursement,0.9763,0.9763
764,claim_to_service_median_ratio,amount_above_service_typical,0.9576,0.9576


High correlation is a **review signal**, not an automatic deletion rule.

Nested fraud-history windows may remain valuable to non-linear models. A reduced specification can later be tested for logistic regression.


## 9. Outlier and finite-value audit

In [12]:
audit_rows = []

for feature in NUMERIC_FEATURES:
    s = claims[feature]

    audit_rows.append({
        "feature": feature,
        "missing_pct": s.isna().mean() * 100,
        "infinite_count": np.isinf(s.dropna()).sum(),
        "min": s.min(),
        "median": s.median(),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
    })

numeric_audit = pd.DataFrame(audit_rows)

numeric_audit.sort_values("missing_pct", ascending=False).head(40)

,feature,missing_pct,infinite_count,min,median,p95,p99,max
20,days_since_same_provider_claim,99.0772,0,0.0004,5.5932,740.3506,"1,034.7992","1,250.0671"
8,days_since_policy_change,91.8978,0,1.0000,45.0000,86.0000,90.0000,90.0000
28,claim_to_customer_avg_ratio,30.1638,0,0.0124,0.8695,5.0731,10.3266,101.6098
37,recent_claim_share_30d_365d,30.1638,0,0.0000,0.0000,1.0000,1.0000,1.0000
18,customer_avg_claim_amount_365d,30.1638,0,8.3100,171.3000,504.1142,813.5929,"3,306.0200"
38,recent_amount_share_30d_365d,30.1638,0,0.0000,0.0000,1.0000,1.0000,1.0000
19,days_since_customer_previous_claim,19.4453,0,0.0001,116.1287,551.8530,818.6718,"1,253.0960"
29,claim_to_provider_avg_ratio,2.1879,0,0.0130,0.8623,2.6328,5.7350,39.3562
25,provider_avg_claim_amount_90d,2.1879,0,16.8100,182.0850,439.5899,583.0845,"2,334.1400"
4,document_count,0.4644,0,1.0000,3.0000,6.0000,6.0000,6.0000


In [13]:
assert numeric_audit["infinite_count"].sum() == 0
print("PASS — no infinite numeric values detected.")

PASS — no infinite numeric values detected.


## 10. Freeze temporal train / validation / test split

In [14]:
TRAIN_END = pd.Timestamp("2025-06-30 23:59:59")
VALIDATION_END = pd.Timestamp("2025-12-31 23:59:59")

train_mask = claims["claim_submission_timestamp"] <= TRAIN_END

validation_mask = (
    (claims["claim_submission_timestamp"] > TRAIN_END)
    & (claims["claim_submission_timestamp"] <= VALIDATION_END)
)

test_mask = claims["claim_submission_timestamp"] > VALIDATION_END

train = claims.loc[train_mask].copy()
validation = claims.loc[validation_mask].copy()
test = claims.loc[test_mask].copy()

split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train),
        "fraud_cases": train[TARGET].sum(),
        "fraud_rate_pct": train[TARGET].mean() * 100,
        "start": train["claim_submission_timestamp"].min(),
        "end": train["claim_submission_timestamp"].max(),
    },
    {
        "split": "validation",
        "rows": len(validation),
        "fraud_cases": validation[TARGET].sum(),
        "fraud_rate_pct": validation[TARGET].mean() * 100,
        "start": validation["claim_submission_timestamp"].min(),
        "end": validation["claim_submission_timestamp"].max(),
    },
    {
        "split": "test",
        "rows": len(test),
        "fraud_cases": test[TARGET].sum(),
        "fraud_rate_pct": test[TARGET].mean() * 100,
        "start": test["claim_submission_timestamp"].min(),
        "end": test["claim_submission_timestamp"].max(),
    },
])

split_summary

,split,rows,fraud_cases,fraud_rate_pct,start,end
0,train,71348,1701,2.3841,2023-01-01 00:22:09,2025-06-30 23:54:11
1,validation,14387,392,2.7247,2025-07-01 00:00:15,2025-12-31 23:34:55
2,test,14176,409,2.8852,2026-01-01 00:17:34,2026-06-30 23:59:59


The newest period is the held-out **test set**.

It must not influence:
- feature decisions;
- hyperparameter tuning;
- threshold selection;
- calibration decisions.


## 11. Preprocessing pipeline

In [15]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=10,
            ),
        ),
    ]
)

boolean_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("boolean", boolean_pipeline, BOOLEAN_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per

### Why this preprocessing?

**Numeric**
- median imputation is robust to skew;
- scaling supports regularized linear models.

**Categorical**
- missingness becomes an explicit category;
- unseen categories are handled safely;
- rare categories can be grouped.

**Boolean / indicators**
- deterministic imputation keeps the representation simple.

Tree-based models may later use a specialized pipeline, but this is a clean common baseline.


## 12. Fit transformations on train only

In [16]:
X_train = train[candidate_features]
y_train = train[TARGET]

X_validation = validation[candidate_features]
y_validation = validation[TARGET]

X_test = test[candidate_features]
y_test = test[TARGET]

preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_validation_transformed = preprocessor.transform(X_validation)

print(f"Raw train shape:        {X_train.shape}")
print(f"Transformed train:      {X_train_transformed.shape}")
print(f"Transformed validation: {X_validation_transformed.shape}")

Raw train shape:        (71348, 57)
Transformed train:      (71348, 107)
Transformed validation: (14387, 107)


In [17]:
feature_names_out = preprocessor.get_feature_names_out()

print(f"Final transformed feature count: {len(feature_names_out)}")
feature_names_out[:60]

Final transformed feature count: 107


array(['service_units', 'claim_amount', 'requested_reimbursement',
       'coverage_limit', 'document_count', 'customer_age',
       'customer_tenure_months', 'policy_tenure_months',
       'days_since_policy_change', 'provider_tenure_months',
       'days_service_to_submission', 'reimbursement_ratio',
       'customer_claims_7d', 'customer_claims_30d', 'customer_claims_90d',
       'customer_claims_365d', 'customer_amount_30d',
       'customer_amount_365d', 'customer_avg_claim_amount_365d',
       'days_since_customer_previous_claim',
       'days_since_same_provider_claim', 'customer_provider_claims_30d',
       'same_service_claims_30d', 'provider_claims_30d',
       'provider_claims_90d', 'provider_avg_claim_amount_90d',
       'service_typical_amount', 'claim_to_service_median_ratio',
       'claim_to_customer_avg_ratio', 'claim_to_provider_avg_ratio',
       'submission_hour', 'submission_dayofweek', 'submission_month',
       'service_dayofweek', 'service_month', 'requested_to_

## 13. Split drift audit

In [18]:
DRIFT_FEATURES = [
    "claim_amount",
    "requested_reimbursement",
    "customer_claims_30d",
    "provider_claims_30d",
    "same_service_claims_30d",
    "claim_to_service_median_ratio",
    "customer_provider_claims_30d",
]

drift_rows = []

for feature in DRIFT_FEATURES:
    train_median = train[feature].median()
    validation_median = validation[feature].median()
    test_median = test[feature].median()

    drift_rows.append({
        "feature": feature,
        "train_median": train_median,
        "validation_median": validation_median,
        "test_median": test_median,
    })

drift_table = pd.DataFrame(drift_rows)
drift_table

,feature,train_median,validation_median,test_median
0,claim_amount,144.4250,143.1400,145.7100
1,requested_reimbursement,112.2500,111.8800,112.7450
2,customer_claims_30d,0.0000,0.0000,0.0000
3,provider_claims_30d,2.0000,2.0000,2.0000
4,same_service_claims_30d,0.0000,0.0000,0.0000
5,claim_to_service_median_ratio,0.9961,1.0003,0.9998
6,customer_provider_claims_30d,0.0000,0.0000,0.0000


## 14. Feature registry

In [19]:
registry = []

for feature in claims.columns:
    if feature == TARGET:
        decision = "TARGET"
        reason = "Prediction target"
    elif feature in IDENTIFIER_COLUMNS:
        decision = "EXCLUDE"
        reason = "Identifier / memorization risk"
    elif feature in LEAKAGE_COLUMNS:
        decision = "EXCLUDE"
        reason = "Synthetic ground truth / target leakage"
    elif feature in SYNTHETIC_PROFILE_COLUMNS:
        decision = "EXCLUDE"
        reason = "Synthetic latent population segment"
    elif feature in RAW_DATE_COLUMNS:
        decision = "TRANSFORM"
        reason = "Calendar features / split timestamp"
    elif feature in candidate_features:
        decision = "INCLUDE"
        reason = "Eligible scoring-time feature"
    else:
        decision = "REVIEW"
        reason = "Not classified"

    registry.append({
        "feature": feature,
        "decision": decision,
        "reason": reason,
    })

feature_registry = pd.DataFrame(registry)

feature_registry["decision"].value_counts()

decision
INCLUDE      57
EXCLUDE      12
TRANSFORM     3
TARGET        1
Name: count, dtype: int64

In [20]:
feature_registry

,feature,decision,reason
0,claim_id,EXCLUDE,Identifier / memorization risk
1,customer_id,EXCLUDE,Identifier / memorization risk
2,policy_id,EXCLUDE,Identifier / memorization risk
3,provider_id,EXCLUDE,Identifier / memorization risk
4,service_category,INCLUDE,Eligible scoring-time feature
5,service_code,INCLUDE,Eligible scoring-time feature
6,service_units,INCLUDE,Eligible scoring-time feature
7,service_date,TRANSFORM,Calendar features / split timestamp
8,claim_submission_date,TRANSFORM,Calendar features / split timestamp
9,claim_submission_timestamp,TRANSFORM,Calendar features / split timestamp


## 15. Modeling contract

In [21]:
print(f"Numeric features:       {len(NUMERIC_FEATURES)}")
print(f"Categorical features:   {len(CATEGORICAL_FEATURES)}")
print(f"Boolean features:       {len(BOOLEAN_FEATURES)}")
print(f"Raw model features:     {len(candidate_features)}")

assert (
    len(set(NUMERIC_FEATURES))
    + len(set(CATEGORICAL_FEATURES))
    + len(set(BOOLEAN_FEATURES))
    == len(candidate_features)
)

print("PASS — every candidate feature belongs to exactly one preprocessing family.")

Numeric features:       42
Categorical features:   7
Boolean features:       8
Raw model features:     57
PASS — every candidate feature belongs to exactly one preprocessing family.


## 16. Final recommendations

### Baseline
Use a **regularized logistic regression** first.

This provides:
- a strong interpretable benchmark;
- a test of how useful engineered features are without complex interactions;
- a reference for judging whether boosting adds meaningful value.

### Non-linear models
Then compare tree-based boosting models.

Expected strengths:
- non-linear thresholds;
- interactions;
- robustness to skewed distributions.

### Imbalance
Accuracy must not drive model selection.

Primary metrics:
- PR-AUC / Average Precision;
- precision;
- recall;
- ROC-AUC as secondary discrimination measure;
- Precision@K;
- Recall@K;
- captured fraud amount at fixed review capacity.

### Calibration
Ranking quality and probability calibration must be evaluated separately.

### Threshold
The operating threshold must be chosen from business constraints, not automatically fixed at 0.5.

### Test set
The 2026 test period remains untouched until the final evaluation notebook.


# Feature Analysis — Completion Checklist

- [ ] Forbidden leakage variables are excluded.
- [ ] Raw identifiers are excluded from the primary model.
- [ ] Raw dates are transformed rather than directly modeled.
- [ ] Missingness indicators are explicit where appropriate.
- [ ] Historical features are confirmed strict-past.
- [ ] New contextual business features are documented.
- [ ] Numeric / categorical / boolean families are frozen.
- [ ] Highly correlated feature groups are reviewed.
- [ ] No infinite values remain.
- [ ] Temporal train / validation / test periods are frozen.
- [ ] Preprocessing is fitted on train only.
- [ ] Final transformed feature names are inspectable.
- [ ] Test data remains unused for model selection.
